# Surface-Based Source Spaces

The Source Space is a distributed dipole model. Brain sources are approximated by dipoles and placed uniformly along the brain surface (more specifically, the white surface — the boundary between gray and white matter). The number of dipoles is adjustable based on the `spacing`. We then eliminate any dipole that is not located within the gray matter, as only the synaptic currents occurring in the gray matter are believed to contribute to EEG.

<img src="../../img/sourcespace.png">

## 📦 Imports

In [1]:
from pathlib import Path
import mindmap as mm
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder.
    - white surface
- `ctx`: Cortical Mask. 
- `cras`: Translation to switch from FreeSurfer space to Subject space.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [2]:
################# Parameters ####################
fconfig = "../../papers/spike/configuration.yml"
sub     = 1
#################################################

# configuration
reader = mm.io.dataset_reader(fconfig)
data   = reader(sub, "ctx", "cras") 

# directories
fs_dir = reader.config["FS_DIR"]
mm_dir = reader.config["MM_DIR"]

DATA SUMMARY
Dataset: spike
Subject: sub-pat01
Retrieved objects:
- cras
- ctx
Missing objects:


## Source Space

#### ⚙️ **<font color=green>Generate Source Space</font>**

📤 **<font color=pink>Outputs</font>**
- `sources`: MNE object containing dipole locations and dipole distances to each other.

🛠️ **<font color=violet>Source space propeties</font>**:
- `spacing`: Spacing in [mm] between dipoles.

🛠️ **<font color=violet>Other options</font>**:
- `n_jobs`: Number of parrallel jobs

In [3]:
########## Parameters ###########
# source space 
spacing = 7

# other options
n_jobs = 10
#################################


# source space
sources = mne.setup_source_space(
    subject=data["sub"],
    spacing=spacing,
    surface="white",
    add_dist=True,      # geodesic distances between sources
    subjects_dir=fs_dir,
    n_jobs=n_jobs,
    verbose="warning",
)
    
# gray matter constraint 
sources = mm.utils.mask_sources(sources, data["ctx"], data["cras"])

print("\033[92mSUMMARY\033[0m")
for src in sources:
    print(f"Source space id: {src['id']}")
    print(f"Type: {src['type']}")
    print(f"Number of sources: {src['nuse']}\n")

SUMMARY
Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 2673

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 2773



🌟 **Visualize source space**

In [11]:
mm.viz.plot_brain(sources, show_src=True, overlay=data["ctx"], cras=data["cras"], overlay_color="gray", overlay_opacity=0.5).show()

Widget(value='<iframe src="http://localhost:43589/index.html?ui=P_0x7e58bc0b3350_7&reconnect=auto" class="pyvi…

#### 💾 Save Source Space

🛠️ **<font color=violet>Options</font>**:
- `overwrite`: if True, overwrites previously saved data.

In [12]:
######### Parameters ############
overwrite = True 
#################################

fsrc   = mm_dir/data['sub']/"src"/f"{data['sub']}_src.fif"

if not fsrc.parent.exists():
    fsrc.parent.mkdir(parents=True)

sources.save(fsrc, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved surface-based source space for subject {data['sub']}\033[0m")

Successfully saved surface-based source space for subject sub-pat01
